In [6]:
import json
import os
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

In [7]:
#postcard_folder_trial = '../data/trial_pix'

postcard_folder = './Images'
json_file = './data/postcard_clip_embeddings.json'

In [8]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 19786.33it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [10]:
# update folder name to postcard_folder - right now set to trial folder for testing

embeddings = {}

for postcard in os.listdir(postcard_folder):
    if postcard.lower().endswith(('.jpg', '.jpeg', '.png')):
        postcard_path = os.path.join(postcard_folder, postcard)
        try:
            with Image.open(postcard_path) as pix:
                # Convert to RGB to avoid issues with grayscale or RGBA images
                pix = pix.convert('RGB')
                inputs = processor(images=pix, return_tensors='pt').to(device)
                
                with torch.no_grad():
                    outputs = model.get_image_features(**inputs)

                # Extract the tensor based on the returned object type
                if hasattr(outputs, 'image_embeds'):
                    features = outputs.image_embeds
                elif hasattr(outputs, 'pooler_output'):
                    features = outputs.pooler_output
                else:
                    features = outputs
                    
                vector = features.squeeze().cpu().tolist()
                embeddings[postcard] = vector
        except Exception as e:
            print(f"Failed to process {postcard}: {e}")

with open(json_file, 'w') as f:
    json.dump(embeddings, f, indent=4)

In [12]:
with open('./data/data_with_geography.json', 'r') as f:
    geo_data = json.load(f)

for item in geo_data:
    file_name = item.get('name')
    # Assign the list vector or a None default if missing
    item['embedding'] = embeddings.get(file_name, None)

with open('./data/data_with__geography_embeddings.json', 'w') as f:
    json.dump(geo_data, f, indent=4)